# Kirschblüte and Weather Correlation
This notebook randomly selects 5 Kirschblüten (sour cherry) stations and 5 long-running weather stations (100+ years history up to the latest date). It then calculates and visualizes the correlation between the blooming day (Jultag) and the spring mean temperature (March-May).

In [ ]:
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
import random

sns.set_theme(style="whitegrid")

# Set a seed for reproducibility
random.seed(42)

In [ ]:
print("Loading Phenology Data...")
df_pheno = pd.read_parquet('../data/processed/kirschbluete_doy.parquet')
unique_kirsch_stations = df_pheno['Stations_id'].unique()

selected_kirsch = random.sample(list(unique_kirsch_stations), 5)
print(f"Selected 5 random Kirschblüten stations: {selected_kirsch}")

df_pheno_sample = df_pheno[df_pheno['Stations_id'].isin(selected_kirsch)].copy()
df_pheno_sample.rename(columns={'Referenzjahr': 'Year'}, inplace=True)

In [ ]:
print("Scanning Weather Station History...")
dataset = ds.dataset('../data/dwd_historical_lake', format='parquet', partitioning='hive')
df_dates = dataset.to_table(columns=['Station_ID', 'Date']).to_pandas()

summary = df_dates.groupby('Station_ID').agg(
    Earliest_Date=('Date', 'min'),
    Latest_Date=('Date', 'max'),
    Days_of_History=('Date', 'count')
).reset_index()

summary['Years_of_History'] = summary['Days_of_History'] / 365.25
youngest_date = summary['Latest_Date'].max()

# Filter: >= 100 years and reporting until the youngest year in the dataset
valid_weather = summary[
    (summary['Years_of_History'] >= 100) &
    (summary['Latest_Date'].dt.year == youngest_date.year)
]

selected_weather = random.sample(list(valid_weather['Station_ID']), 5)
print(f"Selected 5 random 100+ Year Weather Stations: {selected_weather}")

In [ ]:
print("Loading Temperature Data for selected weather stations...")
df_weather = dataset.to_table(
    filter=ds.field('Station_ID').isin(selected_weather),
    columns=['Station_ID', 'Date', 'Mean_Temp_C']
).to_pandas()

df_weather['Year'] = df_weather['Date'].dt.year
df_weather['Month'] = df_weather['Date'].dt.month

# We average the temperature over Spring (March, April, May) which strongly affects bloom
df_spring = df_weather[df_weather['Month'].isin([3, 4, 5])]
yearly_spring_temp = df_spring.groupby(['Station_ID', 'Year'])['Mean_Temp_C'].mean().reset_index()

In [ ]:
print("Pivoting and Merging Datasets...")

# Pivot phenology data (Year vs Station_ID -> Jultag)
pheno_pivot = df_pheno_sample.pivot(index='Year', columns='Stations_id', values='Jultag')
pheno_pivot.columns = [f"Pheno_{c}" for c in pheno_pivot.columns]

# Pivot weather data (Year vs Station_ID -> Mean_Temp_C)
weather_pivot = yearly_spring_temp.pivot(index='Year', columns='Station_ID', values='Mean_Temp_C')
weather_pivot.columns = [f"Weather_{c}" for c in weather_pivot.columns]

# Merge both on Year
merged_df = pd.merge(pheno_pivot, weather_pivot, left_index=True, right_index=True, how='inner')

# Calculate correlation
corr_matrix = merged_df.corr()
cross_corr = corr_matrix.loc[pheno_pivot.columns, weather_pivot.columns]

plt.figure(figsize=(10, 6))
sns.heatmap(cross_corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, fmt=".2f")
plt.title('Correlation: Spring Mean Temp vs Bloom Day (Jultag)', fontsize=14)
plt.xlabel('Weather Stations (Spring Mean Temp °C)', fontsize=12)
plt.ylabel('Kirschblüten Stations (Jultag)', fontsize=12)
plt.tight_layout()
plt.show()